In [1]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
from kp_info_loader_core import InitCore
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [2]:
redis_cli = InitRedis()

In [9]:
temp = pickle.loads(redis_cli.get_data('INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_kline'))
temp.groupby('base_asset').tail(1)

,base_asset,datetime_now,tp_open,tp_high,tp_low,tp_close,LS_open,LS_high,LS_low,LS_close,...,SL_high,SL_low,SL_close,dollar,record_count,tp,scr,atp24h,converted_tp,closed
0,1INCH,2023-10-24 08:07:00,-0.420062,-0.420062,-0.595379,-0.595379,-0.384924,-0.192523,-0.490263,-0.333097,...,-0.490263,-0.787792,-0.630368,1342.5,135,379.0,1.88172,7002444918.70545,381.27,True
1,AAVE,2023-10-24 08:07:00,-0.467999,-0.408986,-0.859213,-0.847472,-0.400253,-0.349902,-0.506399,-0.506399,...,-0.441344,-0.826854,-0.782758,1342.5,135,112400.0,-0.530973,22517140570.480328,113374.125,True
2,ADA,2023-10-24 08:07:00,-0.540975,-0.310632,-0.771482,-0.647386,-0.505454,-0.080454,-0.505454,-0.381737,...,-0.505454,-0.806908,-0.682806,1342.5,135,374.0,-0.266667,19151764489.649059,376.57125,True
3,ALGO,2023-10-24 08:07:00,-0.279896,-0.178658,-0.481759,-0.380930,-0.279896,-0.178658,-0.380930,-0.380930,...,-1.035352,-1.235685,-1.235685,1342.5,135,132.0,0.763359,8760166129.073854,132.50475,True
4,ANKR,2023-10-24 08:07:00,-0.479008,-0.388163,-0.614966,-0.614966,-0.388163,-0.139346,-0.524369,-0.230336,...,-0.524369,-0.909041,-0.614966,1342.5,135,29.3,-1.677852,11596731135.683668,29.4813,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,XLM,2023-10-24 08:07:00,-0.242317,-0.242317,-1.002015,-0.328995,-0.181553,-0.181553,-0.415523,-0.337655,...,-0.838353,-1.070764,-1.002015,1342.5,135,154.0,-0.645161,9227435803.082079,154.52175,True
65,XRP,2023-10-24 08:07:00,-0.673672,-0.234524,-0.673672,-0.307989,-0.104098,-0.104098,-0.454595,-0.289633,...,-0.260336,-0.646312,-0.445117,1342.5,135,727.0,-0.410959,315324483638.200806,729.246,True
66,XTZ,2023-10-24 08:07:00,-0.328607,-0.328607,-0.470589,-0.470589,-0.222348,-0.222348,-0.328607,-0.328607,...,-0.576698,-0.682806,-0.576698,1342.5,135,938.0,-0.635593,3816685272.88707,942.435,True
67,ZIL,2023-10-24 08:07:00,-0.574144,-0.519723,-1.035765,-0.628504,-0.519723,-0.166661,-0.628504,-0.221244,...,-0.628504,-1.089843,-0.682806,1342.5,135,24.4,0.411523,11651009873.768778,24.554325,True


In [ ]:
temp[temp['base_asset']=='ORBS']

In [ ]:
temp = pickle.loads(redis_cli.get_data('INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_kline'))
temp.groupby('base_asset').tail(1)

In [ ]:
temp = pickle.loads(redis_cli.get_data('UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_k'))
temp[temp['base_asset'].isin(['XRP','BTC'])][['base_asset','LS_close','SL_close','dollar']]

In [3]:
logging_dir = "/home/kp_info_loader/"
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
    config = json.load(f)
proc_n = 5
# node = config['node']
node = 'kp_info_loader'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets_dict = config['enabled_markets']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [4]:
core = InitCore(logging_dir, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, enabled_markets_dict, db_dict)

2023-11-01 15:59:46,571 - kp_info_loader - INFO - InitCore|InitCore initiated with proc_n=5
2023-11-01 15:59:46,737 - okx_plug - INFO - okx_plug_logger started.
2023-11-01 15:59:47,023 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-01 15:59:47,279 - binance_plug - INFO - binance_plug_logger started.
2023-11-01 15:59:47,281 - kp_info_loader - INFO - InitCore|update_upbit_spot_info_df thread has started.
2023-11-01 15:59:47,283 - kp_info_loader - INFO - InitCore|update_upbit_spot_ticker_df thread has started.
2023-11-01 15:59:47,286 - kp_info_loader - INFO - InitCore|update_upbit_wallet_status_df thread has started.
2023-11-01 15:59:47,288 - kp_info_loader - INFO - InitCore|update_binance_spot_ticker_df thread has started.
2023-11-01 15:59:47,290 - kp_info_loader - INFO - InitCore|update_binance_spot_info_df thread has started.
2023-11-01 15:59:47,291 - kp_info_loader - INFO - InitCore|update_binance_usd_m_ticker_df thread has started.
2023-11-01 15:59:47,294 - kp_info_loader -

2023-11-01 16:00:19,450 - update_dollar - INFO - fetch_dollar|Dollar price (1357.5 KRW) has been updated.


In [12]:
origin_market_df

,scr,tp,v,atp24h,bp,ap,base_asset,quote_asset,converted_tp,converted_ap,converted_bp
0,-2.071,0.2885,62164263,17961479.709,0.2885,0.2886,1INCH,USDT,391.63875,391.7745,391.63875
1,0.79,82.96,757981.4,61855513.211,82.93,82.94,AAVE,USDT,112618.2,112591.05,112577.475
2,-3.882,0.2872,727171469,213317611.2977,0.2872,0.2873,ADA,USDT,389.874,390.00975,389.874
3,-1.644,0.1077,554734638.2,60345982.4304,0.1077,0.1078,ALGO,USDT,146.20275,146.3385,146.20275
4,0.044,0.02278,579162125,13004805.26906,0.02278,0.02279,ANKR,USDT,30.92385,30.937425,30.92385
...,...,...,...,...,...,...,...,...,...,...,...
71,1.216,0.11903,778463534,94213279.97556,0.11904,0.11905,XLM,USDT,161.583225,161.610375,161.5968
72,3.495,0.5923,2407639357.0,1436446054.7273,0.5921,0.5922,XRP,USDT,804.04725,803.9115,803.77575
73,0.402,0.749,40361128.0,30052452.156,0.748,0.749,XTZ,USDT,1016.7675,1016.7675,1015.41
74,0.992,0.01935,1452314557,27540124.25251,0.01935,0.01936,ZIL,USDT,26.267625,26.2812,26.267625


In [14]:
target_market_df

,tp,scr,atp24h,h52wp,l52wp,ms,mw,tms_x,cd,tms_y,ap,bp,base_asset,quote_asset
0,393.0,-1.503759,5130379688.934793,996.0,300.0,ACTIVE,NONE,1698822180783,KRW-1INCH,1698822191564,393.0,392.0,1INCH,KRW
1,112950.0,1.940433,2804633246.913722,137850.0,63160.0,ACTIVE,NONE,1698822189005,KRW-AAVE,1698822192116,113200.0,112800.0,AAVE,KRW
2,392.0,-1.754386,21743245605.864353,609.0,302.0,ACTIVE,NONE,1698822187960,KRW-ADA,1698822190604,392.0,391.0,ADA,KRW
3,147.0,-1.342282,6405497023.201903,637.0,119.0,ACTIVE,NONE,1698822180792,KRW-ALGO,1698822183296,147.0,146.0,ALGO,KRW
4,31.1,0.322581,11660870093.310333,72.0,18.5,ACTIVE,NONE,1698822180413,KRW-ANKR,1698822189641,31.1,31.0,ANKR,KRW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,162.0,-1.818182,31489180459.117023,238.0,89.4,ACTIVE,NONE,1698822189462,KRW-XLM,1698822189918,163.0,162.0,XLM,KRW
68,807.0,-0.859951,476610939636.154846,1125.0,408.0,ACTIVE,NONE,1698822187586,KRW-XRP,1698822192161,807.0,806.0,XRP,KRW
69,1020.0,-0.487805,2582543189.744977,2055.0,840.0,ACTIVE,NONE,1698822189767,KRW-XTZ,1698822189794,1020.0,1015.0,XTZ,KRW
70,26.4,2.325581,15508893217.974382,48.5,19.3,ACTIVE,NONE,1698822180250,KRW-ZIL,1698822191924,26.4,26.3,ZIL,KRW


2023-11-01 16:04:26,028 - update_dollar - INFO - fetch_dollar|Dollar price (1357.0 KRW) has been updated.


In [8]:
# POSSIBLE quote_assets: USDT, BUSD, BTC, KRW
origin_market = origin_market_code.split('/')[0]
quote_asset_one = origin_market_code.split('/')[1]
target_market = target_market_code.split('/')[0]
quote_asset_two = target_market_code.split('/')[1]

origin_market_df = core.exchange_websocket_dict[origin_market].get_price_df()
origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
target_market_df = core.exchange_websocket_dict[target_market].get_price_df()
target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

shared_base_asset_list = list(set(origin_market_df['base_asset'].values).intersection(set(target_market_df['base_asset'].values)))
origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_base_asset_list)].sort_values('base_asset').reset_index(drop=True)
target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_base_asset_list)].sort_values('base_asset').reset_index(drop=True)

convert_rate = core.convert_asset_rate(origin_market, quote_asset_one, target_market, quote_asset_two)
origin_market_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['tp','ap','bp']] * convert_rate
# target_market_df[['converted_tp','converted_ap','converted_bp']] = target_market_df[['tp','ap','bp']]

# divide by target_market_df[['tp','ap','bp']]
premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
                        origin_market_df[['converted_tp','converted_bp','converted_ap']].values, columns=['tp_premium','LS_premium','SL_premium'])
premium_df['LS_SL_spread'] = premium_df['LS_premium'] - premium_df['SL_premium']
premium_df[['base_asset','quote_asset','tp','ap','bp','scr','atp24h']] = target_market_df[['base_asset','quote_asset','tp','ap','bp','scr','atp24h']]
premium_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['converted_tp','converted_ap', 'converted_bp']]
premium_df.loc[:, ['tp_premium','LS_premium','SL_premium','LS_SL_spread']] = premium_df[['tp_premium','LS_premium','SL_premium','LS_SL_spread']] * 100
premium_df = premium_df.sort_values('atp24h', ascending=False).reset_index(drop=True)
# TEST
premium_df['dollar'] = core.get_dollar_dict()['price']

ValueError: operands could not be broadcast together with shapes (72,3) (76,3) 

2023-11-01 16:03:24,345 - update_dollar - INFO - fetch_dollar|Dollar price (1357.0 KRW) has been updated.


In [5]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
core.get_premium_df(target_market_code, origin_market_code)

2023-11-01 16:00:37,652 - kp_info_loader - ERROR - get_premium_df|Exception occured! Error: operands could not be broadcast together with shapes (72,3) (75,3) , traceback: Traceback (most recent call last):
  File "/home/kp_info_loader/kp_info_loader_core.py", line 319, in get_premium_df
    premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
ValueError: operands could not be broadcast together with shapes (72,3) (75,3) 
, origin_market_df:       scr       tp             v           atp24h       bp       ap  \
0  -1.901    0.289      62148955    17957782.6977   0.2889    0.289   
1   0.669    82.81      752913.5     61433378.812     82.8    82.81   
2  -3.913   0.2873     728300034   213659743.0562   0.2873   0.2874   
3  -1.461   0.1079   556235310.6    60510415.1769   0.1078   0.1079   
4   0.132  0.02282     579702331   13017086.31725  0.02282  0.02283   
..    ...      ...           ...    

ValueError: operands could not be broadcast together with shapes (72,3) (75,3) 

2023-11-01 16:00:50,234 - update_dollar - INFO - fetch_dollar|Dollar price (1357.5 KRW) has been updated.


In [ ]:
origin_market = "UPBIT_SPOT"
target_market = "BINANCE_SPOT"

origin_market_spot_info_df = core.info_dict['binance_spot_info_df']
origin_quote_asset = 'BTC'
target_quote_asset = 'USDT'

if origin_quote_asset == "USD":
    origin_quote_asset = "USDT"
if target_quote_asset == "USD":
    target_quote_asset = "USDT"
if origin_quote_asset == target_quote_asset:
    # return 1
    print(1) # TEST
# origin_market_spot_info_df = self.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
origin_market_spot_info_df = core.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"] # TEST
# First try to find the rate from the info_dict

def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
    df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
    reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
    if len(df) == 1:
        convert_rate = df['lastPrice'].values[0]
    elif len(reverse_df) == 1:
        convert_rate = 1 / reverse_df['lastPrice'].values[0]
    else:
        convert_rate = None
    return convert_rate
convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
if convert_rate is None: # not between coins
    # print("1st convert_rate is None, Not between coins")
    if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
        # convert_rate = self.get_dollar_dict()['price']
        convert_rate = core.get_dollar_dict()['price'] # TEST
    elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
        # convert_rate = 1 / self.get_dollar_dict()['price']
        convert_rate = 1 / core.get_dollar_dict()['price'] # TEST
    elif target_quote_asset == "KRW":
        # convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * self.get_dollar_dict()['price']
        convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * core.get_dollar_dict()['price'] # TEST
    elif origin_quote_asset == "KRW":
        # temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
        temp_convert_rate = core.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset) # TEST
        if temp_convert_rate is not None:
            convert_rate = 1 / temp_convert_rate
        else:
            title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    else:
        # temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, "USDT")
        temp_convert_rate = core.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset) # TEST
        if temp_convert_rate is not None:
            convert_rate = 1 / temp_convert_rate
            # return convert_rate
            print(convert_rate)
        else:
            pass
        title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
        raise Exception(f"Cannot find the convert rate for {title}")
# return convert_rate
print(convert_rate) # TEST

In [ ]:
origin_market_spot_info_df.columns

In [ ]:
origin_market_spot_info_df[['market','base_asset','quote_asset']]#[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]

In [ ]:
temp_convert_rate

In [ ]:
def convert_asset_rate(self, origin_market, origin_quote_asset, target_market, target_quote_asset):
    if origin_quote_asset == "USD":
        origin_quote_asset = "USDT"
    if target_quote_asset == "USD":
        target_quote_asset = "USDT"
    if origin_quote_asset == target_quote_asset:
        return 1
    origin_market_spot_info_df = self.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
    # First try to find the rate from the info_dict

    def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
        df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
        reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
        if len(df) == 1:
            convert_rate = df['lastPrice'].values[0]
        elif len(reverse_df) == 1:
            convert_rate = 1 / reverse_df['lastPrice'].values[0]
        else:
            convert_rate = None
        return convert_rate
    convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
    if convert_rate is None: # not between coins
        # print("1st convert_rate is None, Not between coins")
        if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
            convert_rate = self.get_dollar_dict()['price']
        elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
            convert_rate = 1 / self.get_dollar_dict()['price']
        elif target_quote_asset == "KRW":
            convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * self.get_dollar_dict()['price']
        elif origin_quote_asset == "KRW":
            temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
            if temp_convert_rate is not None:
                convert_rate = 1 / temp_convert_rate
            else:
                title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
                raise Exception(f"Cannot find the convert rate for {title}")
        else:
            title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    return convert_rate

In [ ]:
core.get_market_code_list()

In [ ]:
origin_exchange_code = 'BINANCE_SPOT/BTC'
target_exchange_code = 'UPBIT_SPOT/KRW'

core.get_premium_df(origin_exchange_code, target_exchange_code)

In [ ]:
core.exchange_websocket_dict['UPBIT_SPOT'].check_status(print_result=True)

In [ ]:
core.exchange_websocket_dict['BINANCE_SPOT'].check_status(print_result=True)

In [ ]:
core.exchange_websocket_dict['BINANCE_USD_M'].get_price_df()

In [ ]:
temp = core.exchange_websocket_dict['BINANCE_USD_M'].get_price_df()
temp[temp['base_asset']=='STX']

In [ ]:
temp = core.exchange_websocket_dict['UPBIT_SPOT'].get_price_df()
temp[temp['base_asset']=='STX']

In [ ]:
core.info_dict['binance_spot_ticker_df']['symbol']

In [ ]:
len(core.get_binance_spot_symbol_list())

In [ ]:
core.info_dict['upbit_spot_ticker_df']

In [ ]:
def convert_asset_rate(origin_market, origin_quote_asset, target_market, target_quote_asset):
    if origin_quote_asset == "USD":
        origin_quote_asset = "USDT"
    if target_quote_asset == "USD":
        target_quote_asset = "USDT"
    if origin_quote_asset == target_quote_asset:
        return 1
    origin_market_spot_info_df = core.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
    # First try to find the rate from the info_dict

    def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
        df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
        reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
        if len(df) == 1:
            convert_rate = df['lastPrice'].values[0]
        elif len(reverse_df) == 1:
            convert_rate = 1 / reverse_df['lastPrice'].values[0]
        else:
            convert_rate = None
        return convert_rate
    convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
    if convert_rate is None: # not between coins
        # print("1st convert_rate is None, Not between coins")
        if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
            convert_rate = core.update_dollar_return_dict['price']
        elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
            convert_rate = 1 / core.update_dollar_return_dict['price']
        elif target_quote_asset == "KRW":
            convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * core.update_dollar_return_dict['price']
        elif origin_quote_asset == "KRW":
            temp_convert_rate = convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
            if temp_convert_rate is not None:
                convert_rate = 1 / temp_convert_rate
            else:
                title = f"target_quote_asset: {target_quote_asset}, origin_quote_asset: {origin_quote_asset}"
                raise Exception(f"Cannot find the convert rate for {title}")
        else:
            title = f"target_quote_asset: {target_quote_asset}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    return convert_rate

In [ ]:
exchange_origin = 'BINANCE_USD_M/USDT'
exchange_target = 'UPBIT_SPOT/BTC'
def get_premium_df(self, target_exchange_code, origin_exchange_code):
    # POSSIBLE quote_assets: USDT, BUSD, BTC, KRW
    origin_market = exchange_origin.split('/')[0]
    quote_asset_one = exchange_origin.split('/')[1]
    target_market = exchange_target.split('/')[0]
    quote_asset_two = exchange_target.split('/')[1]

    origin_market_df = self.exchange_websocket_dict[origin_market].get_price_df()
    origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
    target_market_df = self.exchange_websocket_dict[target_market].get_price_df()
    target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

    shared_bass_asset_list = list(set(origin_market_df['base_asset'].values).intersection(set(target_market_df['base_asset'].values)))
    origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
    target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)

    convert_rate = convert_asset_rate(origin_market, quote_asset_one, target_market, quote_asset_two)
    origin_market_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['tp','ap','bp']] * convert_rate
    # target_market_df[['converted_tp','converted_ap','converted_bp']] = target_market_df[['tp','ap','bp']]

    # divide by target_market_df[['tp','ap','bp']]
    premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
                            origin_market_df[['converted_tp','converted_bp','converted_ap']].values, columns=['tp_premium','LS_premium','SL_premium'])
    premium_df['LS_SL_spread'] = premium_df['LS_premium'] - premium_df['SL_premium']
    premium_df[['base_asset','quote_asset','tp','ap','bp','atp24h']] = target_market_df[['base_asset','quote_asset','tp','ap','bp','atp24h']]
    premium_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['converted_tp','converted_ap', 'converted_bp']]
    premium_df.loc[:, ['tp_premium','LS_premium','SL_premium','LS_SL_spread']] = premium_df[['tp_premium','LS_premium','SL_premium','LS_SL_spread']] * 100
    return premium_df

In [ ]:
# POSSIBLE quote_assets: USDT, BUSD, BTC, KRW

exchange_origin = 'BINANCE_USD_M/USDT'
exchange_target = 'UPBIT_SPOT/BTC'

origin_market = exchange_origin.split('/')[0]
quote_asset_one = exchange_origin.split('/')[1]
target_market = exchange_target.split('/')[0]
quote_asset_two = exchange_target.split('/')[1]


origin_market_df = core.exchange_websocket_dict[origin_market].get_price_df()
origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
target_market_df = core.exchange_websocket_dict[target_market].get_price_df()
target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

shared_bass_asset_list = list(set(origin_market_df['base_asset'].values).intersection(set(target_market_df['base_asset'].values)))
origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
# origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].set_index('base_asset')
target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
# target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].set_index('base_asset')

convert_rate = convert_asset_rate(origin_market, quote_asset_one, target_market, quote_asset_two)

origin_market_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['tp','ap','bp']] * convert_rate
# target_market_df[['converted_tp','converted_ap','converted_bp']] = target_market_df[['tp','ap','bp']]

# divide by target_market_df[['tp','ap','bp']]
premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
                          origin_market_df[['converted_tp','converted_bp','converted_ap']].values, columns=['tp_premium','LS_premium','SL_premium'])
premium_df['LS_SL_spread'] = premium_df['LS_premium'] - premium_df['SL_premium']
premium_df[['base_asset','quote_asset','tp','ap','bp','atp24h']] = target_market_df[['base_asset','quote_asset','tp','ap','bp','atp24h']]
premium_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['converted_tp','converted_ap', 'converted_bp']]
premium_df.loc[:, ['tp_premium','LS_premium','SL_premium','LS_SL_spread']] = premium_df[['tp_premium','LS_premium','SL_premium','LS_SL_spread']] * 100
premium_df.sort_values('atp24h', ascending=False).reset_index(drop=True).head(10)


In [ ]:
origin_market_df = core.exchange_websocket_dict[origin_market].get_price_df()
origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
target_market_df = core.exchange_websocket_dict[target_market].get_price_df()
target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

In [ ]:
origin_market_df

In [ ]:
core.exchange_websocket_dict[target_market].get_price_df()

In [ ]:
target_market_df.head()[['base_asset','tp','ap','bp']]

In [ ]:
origin_market_df.head()[['base_asset','tp','ap','bp']]

In [ ]:
origin_market_df.head()

In [ ]:
convert_asset_rate(origin_market="BINANCE_SPOT", origin_quote_asset='BUSD', target_market="UPBIT_SPOT", target_quote_asset='KRW')

In [ ]:
core.update_dollar_return_dict['price']

In [ ]:
upbit_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['UPBIT'].upbit_ticker_dict)).T.reset_index()[['index','tp','scr','atp24h','h52wp','l52wp','ms','mw','tms']]
upbit_orderbook_df = pd.DataFrame(dict(core.exchange_websocket_dict['UPBIT'].upbit_orderbook_dict)).T.reset_index(drop=True)[['cd','tms','obu']]
upbit_orderbook_df['ap'] = upbit_orderbook_df['obu'].apply(lambda x: x[0]['ap'])
upbit_orderbook_df['bp'] = upbit_orderbook_df['obu'].apply(lambda x: x[0]['bp'])
upbit_orderbook_df.drop('obu', axis=1, inplace=True)
upbit_merged_df = pd.merge(upbit_ticker_df, upbit_orderbook_df, left_on='index', right_on='cd', how='inner')
upbit_merged_df['base_asset'] = upbit_merged_df['index'].apply(lambda x: x.split('-')[1])
upbit_merged_df['quote_asset'] = upbit_merged_df['index'].apply(lambda x: x.split('-')[0])
upbit_merged_df.drop('index', axis=1, inplace=True)
upbit_merged_df.loc[:, ['scr','atp24h','h52wp','l52wp','ap','bp']] = upbit_merged_df.loc[:, ['scr','atp24h','h52wp','l52wp','ap','bp']].astype(float)
upbit_merged_df.loc[:, 'scr'] = upbit_merged_df['scr'] * 100
upbit_merged_df.head()

In [ ]:
upbit_merged_df.head()

In [ ]:
core.info_dict['binance_spot_info_df'][['symbol','baseAsset','quoteAsset']].head()

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_SPOT'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_SPOT'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_spot_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df

In [ ]:
binance_merged_df['quote_asset'].unique()

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_USD_M'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_USD_M'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_usdm_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_COIN_M'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_COIN_M'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_coinm_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df